## Modelling Approach

The objective of this stage is to develop regression models for predicting Airbnb nightly prices.

The target variable was log-transformed during preprocessing to reduce the strong right-skewness of Airbnb prices. Therefore, the models are trained to predict the log-transformed price.

The modelling process starts with Linear Regression as a baseline and then evaluates regularized models, Ridge and Lasso, to investigate whether regularization can improve generalization.

The models are evaluated using RMSE, MAE, and R² after converting predictions back to the original price scale.

The main modelling steps are:

1. Train a baseline Linear Regression model.
2. Evaluate model performance using 5-fold cross-validation.
3. Inspect model coefficients.
4. Compare training and test performance.
5. Investigate large prediction errors.
6. Clip extreme predictions to the training target range.
7. Evaluate different Ridge regularization strengths.
8. Use GridSearchCV to tune the Ridge hyperparameter.
9. Evaluate Lasso Regression.
10. Compare the linear models and select the final model.

In [ ]:
#Load your saved preprocessed data
import pandas as pd
import numpy as np

X_train_scaled = pd.read_csv('../data/processed/X_train_scaled.csv')
X_test_scaled = pd.read_csv('../data/processed/X_test_scaled.csv')

y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

## Baseline Model: Linear Regression

Linear Regression was selected as the baseline model because it is a simple and interpretable regression algorithm.

It provides a reference point for evaluating whether additional techniques such as regularization and hyperparameter tuning can improve predictive performance.

The model learns a linear relationship between the input features and the log-transformed Airbnb price.

As this is the baseline model, its performance will be compared with Ridge and Lasso Regression later in the modelling process.

In [ ]:
#Train the baseline model
from sklearn.linear_model import LinearRegression
# establish a simple reference point before testing
#training set is used for model fitting, while the test set is reserved for final evaluation.
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](74,)","[-0.,-0.,-0.,...,-0.,-0., 0.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](74,)","['hosts_time_as_user_years','hosts_time_as_user_months', 'hosts_time_as_host_years',...,'has_wifi','has_kitchen','has_ac']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,5.294
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,74
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(9)


In [ ]:
#Predict on the test set
# few predictions fall outside the training target range
y_pred_log = lr.predict(X_test_scaled)

In [ ]:
#Convert predictions back to real € prices for
# interpretable, stakeholder-facing error metrics.
y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

## 5. Baseline Model Evaluation

The baseline Linear Regression model is evaluated using three metrics:

- **RMSE (Root Mean Squared Error):** Measures the average magnitude of prediction errors while giving greater weight to larger errors.
- **MAE (Mean Absolute Error):** Measures the average absolute prediction error and is easier to interpret in the original price unit.
- **R² (Coefficient of Determination):** Measures the proportion of variance in the target variable explained by the model.

For this project, lower RMSE and MAE indicate better predictive performance, while a higher R² indicates better explanatory performance.

In [ ]:
#Evaluate performance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
r2 = r2_score(y_test_actual, y_pred_actual)

print(f"RMSE: €{rmse:.2f}")
print(f"MAE: €{mae:.2f}")
print(f"R²: {r2:.4f}")

RMSE: €813.08
MAE: €141.12
R²: -10.0534


## Cross-Validation of the Baseline Model

Five-fold cross-validation was used to evaluate the stability of the Linear Regression model.

The training data is divided into five folds. In each iteration, four folds are used for training and one fold is used for validation.

This process is repeated five times so that each fold is used once for validation.

The mean and standard deviation of the cross-validation RMSE provide an indication of the model's average performance and variation across different subsets of the training data.

Cross-validation is performed on the training data so that the test set remains unseen until final evaluation.

In [ ]:
#Cross-validate
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=kf, scoring='neg_root_mean_squared_error')
print(f"CV RMSE (log scale): {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

CV RMSE (log scale): 0.5637 ± 0.0157


## Coefficient Analysis

The coefficients of the Linear Regression model were inspected to understand the relationship between the predictors and the log-transformed target.

The coefficients were sorted according to their absolute magnitude so that the features with the strongest influence on the model's linear predictions could be examined.

Because the features were standardized, coefficient magnitudes can be compared more meaningfully across numerical predictors. However, coefficients should still be interpreted carefully, particularly for one-hot encoded categorical variables.

This analysis helps provide interpretability in addition to predictive evaluation.

In [ ]:
#Inspect coefficients
coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'coefficient': lr.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df.head(15))

                                        feature  coefficient
32                    estimated_occupancy_l365d    -0.003818
17                       maximum_minimum_nights    -0.003206
20                       minimum_nights_avg_ntm    -0.002280
16                       minimum_minimum_nights    -0.001353
30                             availability_eoy     0.001174
25                              availability_90     0.000586
42               calculated_host_listings_count    -0.000540
43  calculated_host_listings_count_entire_homes    -0.000477
14                               minimum_nights    -0.000451
28                        number_of_reviews_ltm    -0.000443
31                         number_of_reviews_ly    -0.000377
24                              availability_60     0.000373
5                           host_listings_count    -0.000221
19                       maximum_maximum_nights     0.000186
23                              availability_30     0.000169


## Training vs Test Performance

Training and test RMSE are compared to identify potential differences between model performance on seen and unseen data.

A substantially lower training error than test error can indicate overfitting, where the model fits the training data well but generalizes poorly to unseen observations.

In this case, the comparison is interpreted together with the cross-validation results rather than using only one train-test split.

In [ ]:
#Check for overfitting
train_pred_log = lr.predict(X_train_scaled)
train_rmse = np.sqrt(mean_squared_error(np.expm1(y_train), np.expm1(train_pred_log)))
print(f"Train RMSE: €{train_rmse:.2f} vs Test RMSE: €{rmse:.2f}")

Train RMSE: €1001.90 vs Test RMSE: €813.08


### Interpretation

The test RMSE was slightly lower than the training RMSE in this particular split.

This does not by itself indicate overfitting. Small differences between training and test performance can occur because of the particular observations assigned to each subset.

The 5-fold cross-validation results are therefore also considered when assessing the stability and generalization of the model.

## Prediction Error Analysis

The largest prediction errors were examined to understand where the Linear Regression model struggled most.

For each test observation, the prediction error and absolute error were calculated.

Sorting by absolute error allows the observations with the largest differences between actual and predicted prices to be investigated.

This analysis provides additional insight into model behaviour that cannot be obtained from aggregate metrics alone.

In [ ]:
# Check actual vs predicted for the worst offenders
results = pd.DataFrame({
    'actual': y_test_actual,
    'predicted': y_pred_actual
})
results['error'] = results['predicted'] - results['actual']
results['abs_error'] = results['error'].abs()

print(results.sort_values('abs_error', ascending=False).head(10))
print("\nMax predicted price:", results['predicted'].max())
print("Min predicted price:", results['predicted'].min())

       actual     predicted         error     abs_error
7240  1846.00  64703.739732  62857.739732  62857.739732
4116  1480.00  21846.982198  20366.982198  20366.982198
3690  1553.00  21092.228655  19539.228655  19539.228655
4733  1780.00  19647.019674  17867.019674  17867.019674
5228  1705.00  14047.617200  12342.617200  12342.617200
1065  1302.00   9110.419089   7808.419089   7808.419089
2315  1255.29   8734.914258   7479.624258   7479.624258
2661  1256.00   8264.786757   7008.786757   7008.786757
5973  1239.00   8154.398248   6915.398248   6915.398248
6594  1234.50   7944.438006   6709.938006   6709.938006

Max predicted price: 64703.739732267975
Min predicted price: 17.41650575002177


## R² on the Logarithmic Scale

R² was also calculated on the log-transformed target to assess how well the Linear Regression model explains variation in the transformed target.

However, the main model comparison is performed using predictions converted back to the original price scale because the practical objective is to predict Airbnb prices in euros.

In [15]:
r2_log = r2_score(y_test, y_pred_log)
print(f"R² (log scale): {r2_log:.4f}")

R² (log scale): 0.4270


## Prediction Clipping

Because the model predicts the log-transformed price, predictions are converted back to the original price scale using `expm1()`.

Due to the exponential transformation, relatively large errors on the log scale can result in very large predicted prices on the original scale.

To reduce the influence of unrealistic extreme predictions, predicted log-prices are constrained to the minimum and maximum log-price values observed in the training data.

The clipped predictions are then converted back to the original price scale and evaluated using RMSE, MAE, and R².

Importantly, this operation is applied only to model predictions. The actual test target values are not modified.

In [ ]:
# Clip predicted log prices to the range seen in training
log_min, log_max = y_train.min(), y_train.max()

y_pred_log_clipped = np.clip(
    y_pred_log,
    log_min,
    log_max
)

y_pred_actual_clipped = np.expm1(y_pred_log_clipped)

rmse_clipped = np.sqrt(
    mean_squared_error(y_test_actual, y_pred_actual_clipped)
)

mae_clipped = mean_absolute_error(
    y_test_actual,
    y_pred_actual_clipped
)

r2_clipped = r2_score(
    y_test_actual,
    y_pred_actual_clipped
)

print(f"Clipped RMSE: €{rmse_clipped:.2f}")
print(f"Clipped MAE: €{mae_clipped:.2f}")
print(f"Clipped R²: {r2_clipped:.4f}")
#Since the target variable was log-transformed, some Linear Regression predictions 
# produced unusually large values when transformed back to the original price scale 
# using the exponential function. 
# To prevent these extreme predictions from disproportionately affecting the 
# evaluation metrics, predicted log-prices were constrained to the observed 
# training-target range before inverse transformation

Clipped RMSE: €212.15
Clipped MAE: €113.70
Clipped R²: 0.2474


## Ridge Regression

Ridge Regression was introduced to investigate whether L2 regularization could improve the baseline Linear Regression model.

The dataset contains many predictors, including one-hot encoded categorical variables. Such features can introduce correlations among predictors and may lead to instability in ordinary Linear Regression coefficients.

Ridge adds an L2 penalty that shrinks model coefficients toward zero without generally eliminating them completely.

The regularization strength is controlled by the `alpha` parameter.

Different alpha values are evaluated to determine whether regularization improves predictive performance.

In [1]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [2]:
alphas = [0.01, 0.1, 1, 10, 100]

## Training and Test Data for Regularized Models

The previously prepared training and test datasets are loaded for the Ridge and Lasso experiments.

The feature matrices contain the scaled predictor variables, while the target vectors contain the log-transformed Airbnb prices.

The same training and test data are used across the linear models to ensure that their performance can be compared consistently.

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

base = r'C:\Users\Umangi\Desktop\FDM project\airbnb-price-prediction\data\processed'

X_train_scaled = pd.read_csv(f'{base}\\X_train_scaled.csv')
X_test_scaled = pd.read_csv(f'{base}\\X_test_scaled.csv')
y_train = pd.read_csv(f'{base}\\y_train.csv').squeeze()
y_test = pd.read_csv(f'{base}\\y_test.csv').squeeze()

print(X_train_scaled.shape, X_test_scaled.shape)

(38333, 74) (9584, 74)


## Comparing Ridge Regularization Strength

The `alpha` parameter controls the strength of Ridge regularization.

A small alpha applies relatively weak regularization, while a larger alpha applies stronger regularization.

The values `0.01`, `0.1`, `1`, `10`, and `100` were tested to observe how increasing regularization affects predictive performance.

For each alpha value, the model is trained on the training data and evaluated on the same held-out test set.

RMSE, MAE, and R² are calculated after converting predictions back to the original price scale.

In [12]:
ridge_results = []

for alpha in alphas:
    
    ridge = Ridge(alpha=alpha)
    
    ridge.fit(X_train_scaled, y_train)
    
    y_pred_log = ridge.predict(X_test_scaled)
    
    # Clip predictions to training target range
    y_pred_log_clipped = np.clip(
        y_pred_log,
        y_train.min(),
        y_train.max()
    )
    
    y_pred_actual = np.expm1(y_pred_log_clipped)
    y_test_actual = np.expm1(y_test)
    
    rmse = np.sqrt(
        mean_squared_error(y_test_actual, y_pred_actual)
    )
    
    mae = mean_absolute_error(
        y_test_actual,
        y_pred_actual
    )
    
    r2 = r2_score(
        y_test_actual,
        y_pred_actual
    )
    
    ridge_results.append({
        "alpha": alpha,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

ridge_results

C:\Users\Umangi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.0118242361626975e-22.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\Umangi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.0118223247687248e-21.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
C:\Users\Umangi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.011817093725695e-20.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=T

[{'alpha': 0.01,
  'RMSE': np.float64(168.24835907142622),
  'MAE': 86.68251706708479,
  'R2': 0.526698850676907},
 {'alpha': 0.1,
  'RMSE': np.float64(168.25601606570092),
  'MAE': 86.68463530704248,
  'R2': 0.5266557697615712},
 {'alpha': 1,
  'RMSE': np.float64(168.31660858221653),
  'MAE': 86.71380771470575,
  'R2': 0.5263147860262516},
 {'alpha': 10,
  'RMSE': np.float64(168.51421298265905),
  'MAE': 86.84506636508242,
  'R2': 0.5252019163192176},
 {'alpha': 100,
  'RMSE': np.float64(169.11911720966165),
  'MAE': 87.1897103471818,
  'R2': 0.5217870963196294}]

## Ridge Regularization Results

The Ridge models produced very similar results across the tested alpha values.

The best test performance among the manually tested values was obtained with `alpha = 0.01`, with approximately:

- RMSE: €168.25
- MAE: €86.68
- R²: 0.5267

As alpha increased, the performance changed only slightly.

This suggests that the predictive performance of the Ridge model is relatively stable across the tested regularization strengths.

## Ridge Hyperparameter Tuning with GridSearchCV

To systematically select the Ridge regularization strength, GridSearchCV was used.

A wider range of alpha values was evaluated:

`0.001, 0.01, 0.1, 1, 10, 100`

Five-fold cross-validation was used to evaluate each candidate value using RMSE on the log-transformed target.

The alpha producing the best cross-validation score was selected automatically.

This approach is preferable to selecting a hyperparameter solely from the test-set results because the test set should remain reserved for final model evaluation.

In [13]:
from sklearn.model_selection import GridSearchCV

ridge = Ridge()
param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(
    ridge, param_grid, cv=5,
    scoring="neg_root_mean_squared_error", n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print("Best alpha:", grid_search.best_params_)
print("Best CV RMSE (log scale):", -grid_search.best_score_)

Best alpha: {'alpha': 0.1}
Best CV RMSE (log scale): 89.94609898752012


C:\Users\Umangi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.0118223247687248e-21.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


## Final Ridge Model Evaluation

The best Ridge estimator identified through 5-fold cross-validation is evaluated on the held-out test set.

The test set is used only after hyperparameter selection to estimate how the optimized model performs on unseen data.

Predictions are converted from the log scale to the original price scale before calculating RMSE, MAE, and R².

In [ ]:
best_ridge = grid_search.best_estimator_
y_pred_log = best_ridge.predict(X_test_scaled)
y_pred_log_clipped = np.clip(y_pred_log, y_train.min(), y_train.max())

y_pred_actual = np.expm1(y_pred_log_clipped)
y_test_actual = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
r2 = r2_score(y_test_actual, y_pred_actual)

print(f"Optimized Ridge — RMSE: €{rmse:.2f}, MAE: €{mae:.2f}, R²: {r2:.4f}")


Optimized Ridge — RMSE: €168.26, MAE: €86.68, R²: 0.5267


### Multicollinearity Observation

An ill-conditioned matrix warning was observed during model fitting, indicating potential numerical instability associated with relationships among the predictors.

The dataset contains one-hot encoded categorical variables and multiple related features, so correlated predictors are possible.

Ridge Regression was therefore considered appropriate to investigate whether coefficient regularization could improve model stability and predictive performance.

However, the relatively small differences across the tested alpha values indicate that changing the regularization strength had only a limited effect on the final predictive metrics.

### RMSE Interpretation

The RMSE is higher than the median Airbnb price, indicating that the model still makes relatively large errors for some listings.

This is expected to some extent because Airbnb prices are highly variable and the dataset contains listings with substantially different characteristics.

Therefore, RMSE is interpreted together with MAE and R² rather than being considered in isolation.

## GridSearchCV with Prediction Clipping

A second hyperparameter tuning experiment was performed using a custom RMSE scorer that clips predictions to the observed training-target range.

This was introduced because the final evaluation procedure also applies prediction clipping.

Using the same prediction treatment during hyperparameter evaluation makes the tuning objective more consistent with the final evaluation procedure.

The same Ridge alpha candidates are evaluated using 5-fold cross-validation.

In [15]:
from sklearn.metrics import make_scorer

def clipped_rmse(y_true, y_pred):
    y_pred_clipped = np.clip(y_pred, y_train.min(), y_train.max())
    return np.sqrt(mean_squared_error(y_true, y_pred_clipped))

clipped_scorer = make_scorer(clipped_rmse, greater_is_better=False)

grid_search = GridSearchCV(
    Ridge(), param_grid={"alpha": [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring=clipped_scorer, n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print("Best alpha:", grid_search.best_params_)
print("Best CV RMSE (log scale, clipped):", -grid_search.best_score_)

Best alpha: {'alpha': 0.01}
Best CV RMSE (log scale, clipped): 0.4131570076340713


C:\Users\Umangi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.0118242361626975e-22.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


## Final Evaluation of Baseline Linear Regression

The baseline Linear Regression model is evaluated using the same prediction clipping and evaluation procedure used for the Ridge and Lasso models.

Using the same evaluation procedure allows the three linear models to be compared fairly.

In [16]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_log = lr.predict(X_test_scaled)
y_pred_log_clipped = np.clip(y_pred_log, y_train.min(), y_train.max())

y_pred_actual = np.expm1(y_pred_log_clipped)
y_test_actual = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
r2 = r2_score(y_test_actual, y_pred_actual)

print(f"Linear Regression (clipped) — RMSE: €{rmse:.2f}, MAE: €{mae:.2f}, R²: {r2:.4f}")

Linear Regression (clipped) — RMSE: €212.15, MAE: €113.70, R²: 0.2474


## Lasso Regression

Lasso Regression was evaluated as a second regularized linear model.

Lasso applies L1 regularization, which can shrink some coefficients exactly to zero.

Therefore, unlike Ridge, Lasso can perform feature selection by removing the contribution of some predictors from the model.

An alpha value of `0.01` was used for this experiment, and the resulting model was evaluated using the same RMSE, MAE, and R² metrics as the other models.

### Lasso Feature Selection

The number of coefficients equal to zero is counted to determine how many features were effectively excluded by Lasso regularization.

This demonstrates the feature-selection capability of L1 regularization.

However, reducing the number of active features does not automatically mean better predictive performance. Therefore, Lasso is compared with Linear Regression and Ridge using the same evaluation metrics.

In [18]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=20000)  # increased from 5000
lasso.fit(X_train_scaled, y_train)

n_zeroed = (lasso.coef_ == 0).sum()
print(f"Features zeroed out by Lasso: {n_zeroed} out of {len(lasso.coef_)}")

y_pred_log = lasso.predict(X_test_scaled)
y_pred_log_clipped = np.clip(y_pred_log, y_train.min(), y_train.max())
y_pred_actual = np.expm1(y_pred_log_clipped)

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
r2 = r2_score(y_test_actual, y_pred_actual)
print(f"Lasso — RMSE: €{rmse:.2f}, MAE: €{mae:.2f}, R²: {r2:.4f}")

C:\Users\Umangi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.127533e+03, tolerance: 2.232e+00
  model = cd_fast.enet_coordinate_descent(


Features zeroed out by Lasso: 45 out of 74
Lasso — RMSE: €180.29, MAE: €94.58, R²: 0.4565
